# **Data retrieval and basic EDA**

## 1. Data retrieval and initial storage

In [ ]:
import pandas as pd

# 📥 Load the external dataset

url = "https://lead-program-assets.s3.eu-west-3.amazonaws.com/M05-Projects/fraudTest.csv"
df = pd.read_csv(url)

In [6]:
df.to_csv("initial_src.csv")

## 2. Data exploration and quick cleaning

In [8]:
df.shape

(555719, 23)

In [13]:
print(df.notnull().sum())


Unnamed: 0               555719
trans_date_trans_time    555719
cc_num                   555719
merchant                 555719
category                 555719
amt                      555719
first                    555719
last                     555719
gender                   555719
street                   555719
city                     555719
state                    555719
zip                      555719
lat                      555719
long                     555719
city_pop                 555719
job                      555719
dob                      555719
trans_num                555719
unix_time                555719
merch_lat                555719
merch_long               555719
is_fraud                 555719
dtype: int64


In [15]:
print(df.dtypes)

Unnamed: 0                 int64
trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object


In [11]:
print(df.columns.tolist())

['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


### Liste des colonnes du dataset

| Colonne               | Description probable                                             | Type                        | À conserver ? |
|-----------------------|------------------------------------------------------------------|-----------------------------|---------------|
| `Unnamed: 0`          | Index ou ID de ligne généré lors de l’export CSV                | 🔢 Numérique (à ignorer)    | ❌             |
| `trans_date_trans_time` | Date et heure de la transaction                                | 📅 DateTime                 | ✅             |
| `cc_num`              | Numéro de carte de crédit (identifiant)                         | 🔒 Identifiant sensible     | ❌             |
| `merchant`            | Nom du commerçant (souvent prefixé par `fraud_`)                | 🏪 Texte (catégorielle)     | ❔             |
| `category`            | Catégorie de la transaction (e.g. food, travel, etc.)           | 📂 Catégorielle             | ✅             |
| `amt`                 | Montant de la transaction                                       | 💰 Numérique continu        | ✅             |
| `first`               | Prénom du porteur de carte                                      | 🧍 Texte (sensible)         | ❌             |
| `last`                | Nom de famille du porteur                                       | 🧍 Texte (sensible)         | ❌             |
| `gender`              | Sexe (`F` ou `M`)                                               | ⚥ Catégorielle              | ✅             |
| `street`              | Adresse du porteur                                              | 🏠 Texte (très sensible)    | ❌             |
| `city`                | Ville du porteur                                                | 🏙️ Texte                   | ❌ redondant code postal  |
| `state`               | État américain (abréviation)                                    | 🏛️ Catégorielle            | ✅             |
| `zip`                 | Code postal                                                     | 🔢 Numérique                | ✅             |
| `lat`                 | Latitude du porteur                                             | 🌐 Géolocalisation          | ❌ redondant code postal             |
| `long`                | Longitude du porteur                                            | 🌐 Géolocalisation          | ❌ redondant code postal            |
| `city_pop`            | Population de la ville                                          | 📊 Numérique                | ✅             |
| `job`                 | Métier du porteur                                               | 👷 Texte libre              | ❔             |
| `dob`                 | Date de naissance                                               | 🎂 Date                    | ❌             |
| `trans_num`           | ID unique de transaction                                        | 🆔 UID technique            | ❌             |
| `unix_time`           | Timestamp (temps Unix)                                          | ⏱️ Numérique               | ❌   redondant trans_date_trans_time          |
| `merch_lat`           | Latitude du commerçant                                          | 🌐 Géolocalisation          | ❌ créer une colonne de distance avant suppression |
| `merch_long`          | Longitude du commerçant                                         | 🌐 Géolocalisation          | ❌ créer une colonne de distance avant suppression|
| `is_fraud`            | ⚠️ Cible : 0 = normal, 1 = fraude                              | 🎯 Binaire                  | ✅             |


In [16]:
# Creation of distance feature

df["distance"] = ((df["lat"] - df["merch_lat"])**2 + (df["long"] - df["merch_long"])**2) ** 0.5


In [19]:
# merchant column analysis

nb_merchants = df["merchant"].nunique()
print(f"Unique merchants : {nb_merchants}")


Unique merchants : 693


In [20]:
# job column analysis

nb_jobs = df["job"].nunique()
print(f"Unique jobs : {nb_jobs}")

Unique jobs : 478


In both cases, there are too many features -> columns will be removed

In [22]:
# keep only relevant columns

cols_to_drop = [
    "Unnamed: 0", "cc_num", "first", "last", "street", "city",
    "job", "dob", "trans_num", "unix_time",
    "lat", "long", "merch_lat", "merch_long",  
    "merchant",                                                  
]

# Création du DataFrame épuré
df_model = df.drop(columns=cols_to_drop)


In [27]:
df_model.head()

,trans_date_trans_time,category,amt,gender,state,zip,city_pop,is_fraud,distance
0,2020-06-21 12:14:25,personal_care,2.86,M,SC,29209,333497,0,0.266004
1,2020-06-21 12:14:33,personal_care,29.84,F,UT,84002,302,0,0.991674
2,2020-06-21 12:14:53,health_fitness,41.28,F,NY,11710,34496,0,0.682970
3,2020-06-21 12:15:15,misc_pos,60.05,M,FL,32780,54767,0,0.250985
4,2020-06-21 12:15:17,travel,3.19,M,MI,49632,1126,0,1.118816


In [28]:
# trans_date_trans_time exploitation

df_model["trans_date_trans_time"] = pd.to_datetime(df_model["trans_date_trans_time"])

df_model["trans_year"] = df_model["trans_date_trans_time"].dt.year
df_model["trans_month"] = df_model["trans_date_trans_time"].dt.month
df_model["trans_day"] = df_model["trans_date_trans_time"].dt.day
df_model["trans_hour"] = df_model["trans_date_trans_time"].dt.hour
df_model["trans_minute"] = df_model["trans_date_trans_time"].dt.minute
df_model["trans_dayofweek"] = df_model["trans_date_trans_time"].dt.dayofweek  # 0 = monday, 6 = sunday
df_model["trans_week"] = df_model["trans_date_trans_time"].dt.isocalendar().week
df_model["trans_is_weekend"] = df_model["trans_dayofweek"].isin([5, 6]).astype(int)

df_model.drop(columns=["trans_date_trans_time"], inplace=True)



In [29]:
df_model.head()

,category,amt,gender,state,zip,city_pop,is_fraud,distance,trans_year,trans_month,trans_day,trans_hour,trans_minute,trans_dayofweek,trans_week,trans_is_weekend
0,personal_care,2.86,M,SC,29209,333497,0,0.266004,2020,6,21,12,14,6,25,1
1,personal_care,29.84,F,UT,84002,302,0,0.991674,2020,6,21,12,14,6,25,1
2,health_fitness,41.28,F,NY,11710,34496,0,0.682970,2020,6,21,12,14,6,25,1
3,misc_pos,60.05,M,FL,32780,54767,0,0.250985,2020,6,21,12,15,6,25,1
4,travel,3.19,M,MI,49632,1126,0,1.118816,2020,6,21,12,15,6,25,1


## 3. Storage for Machine Learning

In [30]:
df_model.to_csv("machine_learning_src.csv")